# Smartphone Addiction: Baseline Modeling

Playground Series S6E8 — sanity baselines, strong native-categorical models, an `_is_missing` out-of-fold ablation, engineered ratio/residual features, and a class-imbalance A/B, each tested directly on cross-validated AUC rather than assumed. Two modes: `evaluate` runs the full comparison; `submission` fits the selected champion on all training data and writes a submission file.

## 1. Config

In [1]:
import os
import time
from typing import Literal

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

RUN_MODE: Literal["evaluate", "submission"] = "evaluate"
CHAMPION_NAME = "hist_gradient_boosting"
NOTEBOOK_VERSION = "baseline-v1"

if RUN_MODE not in {"evaluate", "submission"}:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE}")

# Mode flags: one per experiment block, so each can be toggled without
# commenting code in/out. All evaluation-only work is skipped in submission
# mode, which goes straight to load -> infer -> write.
RUN_V1_SANITY = RUN_MODE == "evaluate"
RUN_LOGISTIC = False  # retired: still numerically unstable even with
                       # solver="saga", penalty="l2", C=0.1, max_iter=2_000
RUN_V2_STRONG = RUN_MODE == "evaluate"
RUN_V2_MISSING_ABLATION = RUN_MODE == "evaluate"
RUN_V3_ENGINEERED = RUN_MODE == "evaluate"
RUN_CLASS_WEIGHT_ABLATION = RUN_MODE == "evaluate"
RUN_SUMMARY = RUN_MODE == "evaluate"

pd.set_option("display.max_columns", 50)

## 2. Data Loading

In [2]:
if os.path.exists("/kaggle/input/competitions/playground-series-s6e8"):
    DATA_DIR = "/kaggle/input/competitions/playground-series-s6e8"
else:
    DATA_DIR = "../data"

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

TARGET = "addicted_label"
NUMERIC_FEATURES = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time",
]
CATEGORICAL_FEATURES = ["gender", "stress_level", "academic_work_impact"]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = train[ALL_FEATURES].copy()
y = train[TARGET].copy()
X_test = test[ALL_FEATURES].copy()

for col in CATEGORICAL_FEATURES:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

print(f"X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")

X: (691369, 12), y: (691369,), X_test: (296302, 12)


## 3. Cross-Validation Helper

In [3]:
results = []  # (name, oof_auc, fold_aucs) for the summary table
oof_store = {}  # name -> oof predictions, for sanity checks / future ensembling

def run_cv(name: str, fit_predict_fold, X_df: pd.DataFrame, y_ser: pd.Series) -> np.ndarray:
    """Run stratified 5-fold CV, print per-fold and overall OOF AUC.

    Args:
        name: label for the results table.
        fit_predict_fold: callable(X_tr, y_tr, X_val) -> val_pred_proba.
        X_df: feature frame.
        y_ser: target series.

    Returns:
        OOF prediction array aligned to X_df's row order.
    """
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X_df))
    fold_aucs = []
    start = time.time()
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_df, y_ser)):
        X_tr, X_val = X_df.iloc[tr_idx], X_df.iloc[val_idx]
        y_tr, y_val = y_ser.iloc[tr_idx], y_ser.iloc[val_idx]
        val_pred = fit_predict_fold(X_tr, y_tr, X_val)
        oof[val_idx] = val_pred
        fold_auc = roc_auc_score(y_val, val_pred)
        fold_aucs.append(fold_auc)
    overall_auc = roc_auc_score(y_ser, oof)
    elapsed = time.time() - start
    print(
        f"{name:40s} OOF AUC={overall_auc:.5f}  "
        f"fold std={np.std(fold_aucs):.5f}  ({elapsed:.0f}s)"
    )
    results.append({
        "name": name,
        "oof_auc": overall_auc,
        "fold_auc_mean": np.mean(fold_aucs),
        "fold_auc_std": np.std(fold_aucs),
        "fold_aucs": fold_aucs,
    })
    oof_store[name] = oof
    return oof

### Model Factory

One factory both the evaluation and submission paths call, so the fitted submission model's estimator configuration (class and hyperparameters) is guaranteed identical to the one the OOF score below was measured on.

In [4]:
def build_model(name: str):
    """Build a configured model without fitting it."""
    if name == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(
            random_state=SEED,
            max_iter=200,
            categorical_features="from_dtype",
        )
    raise ValueError(f"Unknown model: {name}")

## 4. v1 — Sanity Baselines

Constant predictor, logistic regression, and `HistGradientBoostingClassifier` establish the floor and confirm the evaluation pipeline before any tuning. Logistic regression is attempted with a stronger regularization/solver configuration; if it is still numerically unstable, it is retired (`RUN_LOGISTIC = False`) rather than reported as a measured baseline.

In [5]:
if RUN_V1_SANITY:
    # Constant predictor: no ranking signal by construction, AUC = 0.5
    # (not computed via roc_auc_score, which requires score variation);
    # recorded directly as the theoretical floor.
    results.append({
        "name": "v1a_constant", "oof_auc": 0.5,
        "fold_auc_mean": 0.5, "fold_auc_std": 0.0, "fold_aucs": [0.5] * N_FOLDS,
    })
    print(f"{'v1a_constant':40s} OOF AUC=0.50000  (theoretical floor, not fit)")

v1a_constant                             OOF AUC=0.50000  (theoretical floor, not fit)


In [6]:
if RUN_V1_SANITY and RUN_LOGISTIC:
    def fit_predict_logreg(X_tr, y_tr, X_val):
        pipe = Pipeline([
            ("prep", ColumnTransformer([
                ("num", Pipeline([
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]), NUMERIC_FEATURES),
                ("cat", Pipeline([
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]), CATEGORICAL_FEATURES),
            ])),
            ("clf", LogisticRegression(
                solver="saga", penalty="l2", C=0.1, max_iter=2_000,
                random_state=SEED, n_jobs=-1,
            )),
        ])
        pipe.fit(X_tr, y_tr)
        return pipe.predict_proba(X_val)[:, 1]

    _ = run_cv("v1b_logistic_regression", fit_predict_logreg, X, y)

In [7]:
if RUN_V1_SANITY:
    def fit_predict_hgb(X_tr, y_tr, X_val):
        model = build_model(CHAMPION_NAME)
        model.fit(X_tr, y_tr)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v1c_hist_gradient_boosting", fit_predict_hgb, X, y)

v1c_hist_gradient_boosting               OOF AUC=0.95733  fold std=0.00076  (937s)


**Insight:** the constant predictor's AUC=0.5 confirms the floor; HGB's actual OOF AUC (see the summary table in Section 9) confirms the pipeline is producing genuine ranking signal well above that floor before any tuning. Logistic regression is retired here (see above) rather than included in the comparison.

## 5. v2 — Strong Models (Native Categorical)

In [8]:
if RUN_V2_STRONG:
    def fit_predict_lgbm(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1,
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    lgbm_oof = run_cv("v2a_lightgbm_native_cat", fit_predict_lgbm, X, y)

v2a_lightgbm_native_cat                  OOF AUC=0.95480  fold std=0.00068  (944s)


In [9]:
if RUN_V2_STRONG:
    def catboost_ready(df: pd.DataFrame) -> pd.DataFrame:
        # CatBoost's pandas-Categorical cat_features path rejects NaN
        # directly ("cat_features must be integer or string ... NaN
        # values should be converted to string") -- give it an explicit
        # "missing" string category instead, still distinct from every
        # real level, so this is native-missing-handling in spirit even
        # though LightGBM/HGB can take the NaN itself.
        out = df.copy()
        for col in CATEGORICAL_FEATURES:
            out[col] = out[col].astype("object").fillna("missing").astype(str)
        return out

    def fit_predict_catboost(X_tr, y_tr, X_val):
        model = CatBoostClassifier(
            random_seed=SEED, iterations=200, depth=6, learning_rate=0.05,
            cat_features=CATEGORICAL_FEATURES, verbose=False,
        )
        model.fit(catboost_ready(X_tr), y_tr)
        return model.predict_proba(catboost_ready(X_val))[:, 1]

    catboost_oof = run_cv("v2b_catboost_native_cat", fit_predict_catboost, X, y)

v2b_catboost_native_cat                  OOF AUC=0.94190  fold std=0.00063  (1839s)


**Insight:** compare against v1's sanity baselines — native categorical + native missing-value handling should clear the HGB floor if the tree ensembles are extracting more signal than a single boosting pass on ordinal-ish encodings.

## 6. `_is_missing` Indicator Flags — OOF Ablation

The marginal analysis during EDA found no strong target signal in missingness, but explicitly did not rule out a conditional effect. This is the actual test — LightGBM with vs. without explicit `_is_missing` columns alongside native NaN handling, same model/fold setup as v2a for a clean comparison.

In [10]:
if RUN_V2_MISSING_ABLATION:
    X_with_flags = X.copy()
    for col in ALL_FEATURES:
        X_with_flags[f"{col}_is_missing"] = X[col].isna().astype(int)

    def fit_predict_lgbm_flags(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1,
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v2c_lightgbm_plus_missing_flags", fit_predict_lgbm_flags, X_with_flags, y)

v2c_lightgbm_plus_missing_flags          OOF AUC=0.95480  fold std=0.00068  (13s)


**Insight:** compare `v2a_lightgbm_native_cat` vs. `v2c_lightgbm_plus_missing_flags` OOF AUC — this is the direct answer to whether `_is_missing` flags earn their place, not the marginal EDA table.

## 7. v3 — Engineered Features

EDA-informed ratio/residual features among the three strongest predictors, computed identically on train and test, target-free (no leakage from the target into feature construction):

In [11]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add EDA-informed ratio/residual features. Target-free, safe to
    compute once outside the CV loop."""
    out = df.copy()
    out["social_to_screen_ratio"] = df["social_media_hours"] / df["daily_screen_time_hours"].replace(0, np.nan)
    out["gaming_to_screen_ratio"] = df["gaming_hours"] / df["daily_screen_time_hours"].replace(0, np.nan)
    out["time_budget_residual"] = 24 - (
        df["sleep_hours"] + df["work_study_hours"] + df["daily_screen_time_hours"]
    )
    out["weekend_escalation"] = df["weekend_screen_time"] - df["daily_screen_time_hours"]
    return out

ENGINEERED_FEATURES = [
    "social_to_screen_ratio", "gaming_to_screen_ratio",
    "time_budget_residual", "weekend_escalation",
]

if RUN_V3_ENGINEERED:
    X_engineered = add_engineered_features(X)

    def fit_predict_lgbm_engineered(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1,
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v3_lightgbm_plus_engineered", fit_predict_lgbm_engineered, X_engineered, y)

v3_lightgbm_plus_engineered              OOF AUC=0.95533  fold std=0.00062  (12s)


**Insight:** compare `v3_lightgbm_plus_engineered` against `v2a_lightgbm_native_cat` — ratios/residuals of features a tree ensemble can already split on nonlinearly are not guaranteed to help.

## 8. Class-Imbalance A/B

AUC is rank-based, so `class_weight` mainly affects optimizer dynamics rather than the final ranking — tested directly rather than assumed either way.

In [12]:
if RUN_CLASS_WEIGHT_ABLATION:
    def fit_predict_lgbm_balanced(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1, class_weight="balanced",
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v2d_lightgbm_class_weight_balanced", fit_predict_lgbm_balanced, X, y)

v2d_lightgbm_class_weight_balanced       OOF AUC=0.95463  fold std=0.00058  (11s)


**Insight:** compare against `v2a_lightgbm_native_cat` (unweighted) — expect little to no AUC change either way since AUC only depends on score ranking.

## 9. Summary and Candidate Sanity Checks

In [13]:
if RUN_SUMMARY:
    summary = pd.DataFrame(results)[["name", "oof_auc", "fold_auc_mean", "fold_auc_std"]]
    summary = summary.sort_values("oof_auc", ascending=False).reset_index(drop=True)
    display(summary)

,name,oof_auc,fold_auc_mean,fold_auc_std
0,v1c_hist_gradient_boosting,0.957333,0.957334,0.000764
1,v3_lightgbm_plus_engineered,0.955329,0.955332,0.000623
2,v2c_lightgbm_plus_missing_flags,0.954804,0.954806,0.000682
3,v2a_lightgbm_native_cat,0.954800,0.954802,0.000675
4,v2d_lightgbm_class_weight_balanced,0.954631,0.954633,0.000584
5,v2b_catboost_native_cat,0.941899,0.941903,0.000631
6,v1a_constant,0.500000,0.500000,0.000000


In [14]:
def candidate_sanity_checks(name: str, oof_pred: np.ndarray, y_true: pd.Series) -> dict:
    """Sanity-check predictions without assuming a classification threshold
    AUC optimization doesn't make."""
    finite_in_range = bool(np.all(np.isfinite(oof_pred)) and np.all((oof_pred >= 0) & (oof_pred <= 1)))
    n_unique = int(pd.Series(oof_pred).nunique())
    overall_auc = roc_auc_score(y_true, oof_pred)
    return {
        "name": name,
        "finite_in_[0,1]": finite_in_range,
        "n_unique_predictions": n_unique,
        "prediction_range": (float(oof_pred.min()), float(oof_pred.max())),
        "overall_oof_auc": overall_auc,
    }

if RUN_SUMMARY:
    best_name = summary.iloc[0]["name"] if summary.iloc[0]["name"] != "v1a_constant" else summary.iloc[1]["name"]
    checks = candidate_sanity_checks(best_name, oof_store[best_name], y)
    display(pd.Series(checks))

name                       v1c_hist_gradient_boosting
finite_in_[0,1]                                  True
n_unique_predictions                           686565
prediction_range        (1.7233597543566132e-79, 1.0)
overall_oof_auc                              0.957333
dtype: object

**Insight:** the leading candidate (excluding the constant floor) passes basic sanity (finite, bounded, non-degenerate predictions) before being considered for further tuning.

## 10. Next Moves

The strongest configuration here (currently `v1c_hist_gradient_boosting`, an untuned sanity baseline that already beats the untuned "strong models" — see Section 4) is the current champion (`CHAMPION_NAME`), used by the submission path below. Promotion-gate thresholds for future tuning should be derived from this notebook's fold-to-fold OOF AUC std (Section 9), not a borrowed number.

## 11. Submission

Builds a schema-safe submission from the champion model (`build_model(CHAMPION_NAME)`, the same factory Section 4 uses for evaluation), fit on all training rows. Only runs in submission mode (`RUN_MODE = "submission"`); evaluation mode stops above.

In [15]:
def fit_champion_and_predict(
    model_name: str,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
) -> np.ndarray:
    """Fit the selected configuration on all training rows."""
    model = build_model(model_name)
    model.fit(X_train, y_train)
    return model.predict_proba(X_test)[:, 1]


def build_submission(
    sample: pd.DataFrame,
    test_ids: pd.Series,
    predictions: np.ndarray,
) -> pd.DataFrame:
    """Create a schema-safe submission in test-row order."""
    submission = sample.copy()
    submission["id"] = test_ids.to_numpy()
    submission["addicted_label"] = predictions
    if submission.columns.tolist() != ["id", "addicted_label"]:
        raise ValueError("Unexpected submission columns")
    if not submission["id"].equals(test_ids.reset_index(drop=True)):
        raise ValueError("Submission ID order mismatch")
    if not np.isfinite(predictions).all():
        raise ValueError("Non-finite predictions")
    if not ((predictions >= 0.0) & (predictions <= 1.0)).all():
        raise ValueError("Predictions outside [0, 1]")
    return submission


if RUN_MODE == "submission":
    predictions = fit_champion_and_predict(CHAMPION_NAME, X, y, X_test)
    submission = build_submission(sample, test["id"], predictions)
    OUTPUT_PATH = (
        "/kaggle/working/submission.csv"
        if os.path.exists("/kaggle/working")
        else "../submission.csv"
    )
    submission.to_csv(OUTPUT_PATH, index=False)
    print(f"Wrote {OUTPUT_PATH}: {submission.shape}")